In [6]:
import os, re
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.tools import create_retriever_tool # 벡터 검색을 ReAct 에이전트의 도구로 변환
from langchain_core.prompts import PromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import Dict, List, Optional, Tuple


load_dotenv("./.env")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# 임베딩 설정
embedding = OpenAIEmbeddings()

# PDF 문서를 벡터DB로 변환하고 이를 검색할 수 있는 retriever 생성
def create_pdf_retriever(
        pdf_path:str,
        persist_directory:str,
        embedding_model:OpenAIEmbeddings,
        chunk_size:int =512,
        chunk_overlap:int = 0
) -> Chroma.as_retriever:
    
    loader=PyMuPDFLoader(pdf_path)
    data = loader.load()

    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    doc_splits = text_splitter.split_documents(data)

    vectorstore = Chroma.from_documents(
        persist_directory=persist_directory,
        documents=doc_splits,
        embedding=embedding_model
    )

    return vectorstore.as_retriever()

retriever_jpn = create_pdf_retriever(
    pdf_path="./Data/ict_japan_2024.pdf",
    persist_directory='db_ict_policy_jpn_2024',
    embedding_model=embedding
)

retriever_usa = create_pdf_retriever(
    pdf_path="./Data/ict_usa_2024.pdf",
    persist_directory='db_ict_policy_usa_2024',
    embedding_model=embedding
)

In [7]:
# create_retriver_tool: retriever 객체를 ReAct 에이전트가 활용할 수 있는 형태의 도구로 변환,
# description에는 검색기의 상세한 용도를 작성해야함. ReAct에이전트가 보고, 사용자의 질문에 따라서 도구를 선택
jpn_engine = create_retriever_tool(
    retriever=retriever_jpn,
    name="jpn_ict",
    description="일본의 ICT 시장동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요.",
)

usa_engine = create_retriever_tool(
    retriever=retriever_usa,
    name="usa_ict",
    description="미국의 ICT 시장동향 정보를 제공합니다. 미국 ICT와 관련된 질문은 해당 도구를 사용하세요.",
)

tools = [jpn_engine, usa_engine]
tool_map: Dict[str, object] = {t.name: t for t in tools}


In [8]:
react_template = '''다음 질문에 최선을 다해 답변하세요. 당신은 다음 도구들에 접근할 수 있습니다:

{tools}

다음 형식을 사용하세요:

Question: 답변해야 하는 입력 질문
Thought: 무엇을 할지 항상 생각하세요
Action: 취해야 할 행동, [{tool_names}] 중 하나여야 합니다. 리스트에 있는 도구 중 1개를 택하십시오.
Action Input: 행동에 대한 입력값
Observation: 행동의 결과
... (이 Thought/Action/Action Input/Observation의 과정이 N번 반복될 수 있습니다)
Thought: 이제 최종 답변을 알겠습니다
Final Answer: 원래 입력된 질문에 대한 최종 답변 (한글로 작성하십시오.)

## 추가적인 주의사항
- 반드시 [Thought -> Action -> Action Input -> Observation] 순서를 준수하십시오. 항상 Action 전에는 Thought가 먼저 나와야 합니다.
- 최종 답변에는 최대한 많은 내용을 포함하십시오.
- 한 번의 검색으로 해결되지 않을 것 같다면 문제를 분할하여 푸는 것도 고려하십시오.
- 정보가 취합되었다면 불필요하게 사이클을 반복하지 마십시오.
- 묻지 않은 정보를 찾으려고 도구를 사용하지 마십시오.

시작하세요!

Question: {input}
{agent_scratchpad}'''

prompt = PromptTemplate.from_template(react_template)

In [9]:
# 도구 목록을 프롬프트에 삽입할 수 있는 문자열 형태로 변환하는 용도로 사용
def _format_tools_for_prompt(ts: List[object]) -> Tuple[str, str]:
    lines, names = [], []
    for t in ts:
        names.append(t.name)
        desc = getattr(t, "description","") # getattr: 객체에서 속성을 꺼내는 함수
        lines.append(f"{t.name}:{desc}")
    return "\n".join(lines), ", ".join(names)

# LLM에 실제로 전달할 최종 프롬프트 문자열을 만드는 함수
def _render_prompt(user_input: str, scratchpad: str) -> str:
    tools_str, tool_names = _format_tools_for_prompt(tools)
    return prompt.format(
        tools=tools_str,
        tool_names=tool_names,
        input=user_input,
        agent_scratchpad=scratchpad # scratchpad: ReAct 루프가 반복될 때마다 이전 단계의 Thought, Action, Observatoin 기록이 누적된 문자열
    )

llm = ChatOpenAI(model='gpt-4.1', temperature=0)

In [12]:
'''
ACTION_RE = re.compile(r"^Action\s*:\s*(?P.+?)\s*$", re.MULTILINE) 
ACTION_INPUT_RE = re.compile(r"^Action Input\s*:\s*(?P.+?)\s*$", re.MULTILINE) 
FINAL_ANSWER_RE = re.compile(r"Final Answer\s*:\s*(?p[\s\S]+)$", re.IGNORECASE)
'''
ACTION_RE = re.compile(r"^Action\s*:\s*(?P<action>.+?)\s*$", re.MULTILINE) # Action: 뒤의 내용 추출
ACTION_INPUT_RE = re.compile(r"^Action Input\s*:\s*(?P<input>.+?)\s*$", re.MULTILINE) # Action Input: 뒤의 내용 추출
FINAL_ANSWER_RE = re.compile(r"Final Answer\s*:\s*(?P<final>[\s\S]+)$", re.IGNORECASE) # Final Answer: 뒤의 내용 추출

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Dohy\AppData\Local\Temp\ipykernel_2168\3725805283.py:2: SyntaxWarning: invalid escape sequence '\s'
  ACTION_RE = re.compile(r"^Action\s*:\s*(?P.+?)\s*$", re.MULTILINE)


In [ ]:
# LLM 응답 텍스트를 파싱하여 다음에 취할 행동을 판별하는 용도
def _parse_action_and_input(text:str)->Tuple[Optional[str], Optional[str]]:
    m_final = FINAL_ANSWER_RE.search(text)
    if m_final:
        return "__FINAL__", m_final.group("final").strip() # __FINAL__은 루프를 종료하라는 신호
    m_act = ACTION_RE.search(text)
    m_in = ACTION_INPUT_RE.serach(text)
    if m_act and m_in:
        return m_act.group("tool").strip(), m_in.group("input").strip()
    return None, None

In [ ]:
# 도구 실행 결과를 문자열로 변환
def _observation_to_text(observation_obj)->str:
    if isinstance(observation_obj, list):
        def doc_to_str(d):
            try:
                meta = getattr(d, "metadata", {}) or {}
                src = meta.get("source") or meta.get("file_path") or ""
                txt = getattr(d, "page_content","")
                if len(txt)>500:
                    txt = txt[:500] + "..."
                return f"[source={src}] {txt}"
            except Exception:
                return str(d)
            
        return "\n".join(doc_to_str(d) for d in observation_obj[:5])
    return str(observation_obj)

In [ ]:
# ReAct 에이전트의 메인 실행 루프, Thought → Action → Observation 사이클을 반복하는 용도로 사용
def run_react(user_input:str, max_iters:int=8) -> Dict[str,str]:
    scratchpad=""
    for _ in range(max_iters):
        rendered = _render_prompt(user_input, scratchpad)
        resp = llm.invoke(rendered)
        text = resp.content if hasattr(resp, "content") else str(resp)

        tool, action_input = _parse_action_and_input(text)
        if tool is None:
            hint = "\n[파싱안내] 형식을 엄격히 따르세요. 반드시 'Action:'과 'Action Input:'을 한 줄씩 제공하십시오.\n"
            scratchpad += f"{text}\n{hint}"
            continue
        if tool == "__FINAL__":
            final_anser = action_input
            return {'output':final_anser, "log":scratchpad + '\n' + text}
        
        if tool not in tool_map:
            observation = f"[에러] 존재하지 않는 도구입니다: {tool}"
            scratchpad += f"{text}\nObservation: {observation}\n"
            continue

        try:
            observation_obj = tool_map[tool].invoke(action_input)
            observation = _observation_to_text(observation_obj)
            scratchpad += f"{text}\nObservation: {observation}\n"
        except Exception as e:
            scratchpad += f"{text}\nObservation: [도구 실행 오류] {e}\n"

    return {
        'output': "반복 한도를 초과했습니다. 질문을 더 구체화해 주세요.",
        'log':scratchpad,
    }

In [ ]:
result = run_react("한국과 미국의 ICT 기관 협력 사례")
print("최종 답변:", result["output"])
print("\n=== 실행 로그 ===\n")
print(result["log"])